In [ ]:
# a lot of ignore warnings were creating noisy outputs, so filter these away.
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

In [ ]:
# Load BESSTIE dataset and view overall stats
from datasets import load_dataset
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

dataset = load_dataset("surrey-nlp/BESSTIE-CW-26")

print(dataset)

training_dataframe = dataset['train'].to_pandas()
print(training_dataframe['Sentiment'].value_counts())
print(training_dataframe['Sarcasm'].value_counts())
print(training_dataframe['variety'].value_counts())




In [ ]:
# What is the optimal number to use for max_features?
test_dataframe = dataset['test'].to_pandas()

for max_features in [1000, 1500, 2000, 5000, 10000, 20000]:
    print(f"\n{'='*55}\nmax_features = {max_features}\n{'='*55}")

    vectorizer = TfidfVectorizer(max_features=max_features)
    X_train = vectorizer.fit_transform(training_dataframe['text'])
    X_test = vectorizer.transform(test_dataframe['text'])

    # Sentiment
    lr_sentiment = LogisticRegression(max_iter=1000)
    lr_sentiment.fit(X_train, training_dataframe['Sentiment'])
    print("Sentiment:")
    print(classification_report(test_dataframe['Sentiment'], lr_sentiment.predict(X_test)))

    # Sarcasm
    lr_sarcasm = LogisticRegression(max_iter=1000)
    lr_sarcasm.fit(X_train, training_dataframe['Sarcasm'])
    print("Sarcasm:")
    print(classification_report(test_dataframe['Sarcasm'], lr_sarcasm.predict(X_test)))

Accuracy is the wrong metric for our purposes becuase for sarcasm the dataset is highly imbalanced. The 0.86 accuracy is misleading. The precision and recall for sarcasm is 0.00. The macro score is more useful.

1,500 is the smallest vocabulary found with the highest macro-F1. The below explores the effects of applying combinations of bigrams/unigrams and with and without sublinear TF


In [ ]:
from sklearn.metrics import f1_score

configs = [
    {"name": "unigrams, default TF",   "ngram_range": (1,1), "sublinear_tf": False},
    {"name": "unigrams, sublinear TF", "ngram_range": (1,1), "sublinear_tf": True},
    {"name": "bigrams, default TF",    "ngram_range": (1,2), "sublinear_tf": False},
    {"name": "bigrams, sublinear TF",  "ngram_range": (1,2), "sublinear_tf": True},
]

results = []
for max_features in [1000, 1500, 2000, 5000, 10000, 20000]:
    for config in configs:
        vectorizer = TfidfVectorizer(
            max_features=max_features,
            ngram_range=config["ngram_range"],
            sublinear_tf=config["sublinear_tf"],
        )
        X_train = vectorizer.fit_transform(training_dataframe['text'])
        X_test = vectorizer.transform(test_dataframe['text'])

        lr_sent = LogisticRegression(max_iter=1000).fit(X_train, training_dataframe['Sentiment'])
        lr_sarc = LogisticRegression(max_iter=1000).fit(X_train, training_dataframe['Sarcasm'])

        results.append({
            "max_features": max_features,
            "config": config["name"],
            "sent_macro_f1": f1_score(test_dataframe['Sentiment'], lr_sent.predict(X_test), average='macro'),
            "sarc_macro_f1": f1_score(test_dataframe['Sarcasm'], lr_sarc.predict(X_test), average='macro'),
        })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

The best result is unigrams with sublinear TF at max_features=20,000 (0.8373).

At every vocabulary size, bigrams are slightly worse than unigrams.
This finding is consistent with the commentary by Bekkerman and Allan, 2003 regarding Reuters and 20 Newsgroups datasets. They found at best the use of bigrams produces an improvement which is "statistically insignificant".
Using Bigrams in Text Categorization by Bekkerman and Allan, 2003
https://ciir.cs.umass.edu/pubfiles/ir-408.pdf

Next we apply Logistic Regression with balanced class weight.

In [ ]:
# Balanced class weight
lr_sarcasm = LogisticRegression(max_iter=1000, class_weight='balanced')
lr_sarcasm.fit(X_train, training_dataframe['Sarcasm'])
sarcasm_predictions = lr_sarcasm.predict(X_test)
print("Test dataset Logistic Regression Sarcasm metrics with balanced class weight")
print(classification_report(test_dataframe['Sarcasm'], sarcasm_predictions))

The macro F1 score substantially increases from 0.46 to 0.62

The accuracy and weighted averages are slightly lower, but the macro avg is the most important metric.

Next we compare the balanced class with our own class weight ratio sweep.

For details on class weights see "Logistic Regression in Rare Events Data" by King & Zeng, 2001

In [ ]:
from sklearn.metrics import precision_score, recall_score

vectorizer = TfidfVectorizer(max_features=10000, sublinear_tf=True)
X_train = vectorizer.fit_transform(training_dataframe['text'])
X_test = vectorizer.transform(test_dataframe['text'])

# examples from class 0 (sentiment) count with weight 1 in all cases. For examples from class 1 (sarcasm) experiment with values 2, 3, 4, 5, 6 and 8.
results = []
for weight in ['balanced', {0: 1, 1: 2}, {0: 1, 1: 3}, {0: 1, 1: 4}, {0: 1, 1: 5}, {0: 1, 1: 6}, {0: 1, 1: 8}]:
    lr = LogisticRegression(max_iter=1000, class_weight=weight)
    lr.fit(X_train, training_dataframe['Sarcasm'])
    preds = lr.predict(X_test)
    results.append({
        'weight': str(weight),
        'macro_f1': f1_score(test_dataframe['Sarcasm'], preds, average='macro'),
        'sarc_f1': f1_score(test_dataframe['Sarcasm'], preds, pos_label=1),
        'sarc_precision': precision_score(test_dataframe['Sarcasm'], preds, pos_label=1),
        'sarc_recall': recall_score(test_dataframe['Sarcasm'], preds, pos_label=1),
    })

import pandas as pd
pd.DataFrame(results)

The balanced class has the highest recall and f1 scores for sarcasm, confirming it is essentially optimal. The results demonstrate the precision/recall tradeoff.

In [ ]:
from sklearn.svm import LinearSVC

# Sentiment
svm_sentiment = LinearSVC(max_iter=10000)
svm_sentiment.fit(X_train, training_dataframe['Sentiment'])
print("Test dataset SVM Sentiment metrics:")
print(classification_report(test_dataframe['Sentiment'], svm_sentiment.predict(X_test)))

# Sarcasm
svm_sarcasm = LinearSVC(max_iter=10000, class_weight='balanced')
svm_sarcasm.fit(X_train, training_dataframe['Sarcasm'])
print("Test dataset SVM Sarcasm metrics:")
print(classification_report(test_dataframe['Sarcasm'], svm_sarcasm.predict(X_test)))

In [ ]:
!pip install --upgrade transformers

RoBERTa sentiment analysis

In [ ]:
# this code is sentiment specific and can be removed later because what follows is a refactored version of this code for sentiment and sarcasm
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
import numpy as np

model_name = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    return tokenizer(batch['text'], padding='max_length', truncation=True, max_length=128)

tokenized_train = dataset['train'].map(tokenize, batched=True)
tokenized_test = dataset['test'].map(tokenize, batched=True)
tokenized_val = dataset['validation'].map(tokenize, batched=True)

tokenized_train = tokenized_train.map(lambda examples: {'labels': [int(s) for s in examples['Sentiment']]}, batched=True, remove_columns=['Sentiment'])
tokenized_test = tokenized_test.map(lambda examples: {'labels': [int(s) for s in examples['Sentiment']]}, batched=True, remove_columns=['Sentiment'])
tokenized_val = tokenized_val.map(lambda examples: {'labels': [int(s) for s in examples['Sentiment']]}, batched=True, remove_columns=['Sentiment'])

tokenized_train.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])
tokenized_test.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])
tokenized_val.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2, problem_type="single_label_classification")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    from sklearn.metrics import classification_report
    print(classification_report(labels, preds))
    from sklearn.metrics import f1_score
    return {'macro_f1': f1_score(labels, preds, average='macro')}



trainer.train()

Test set evaluation

In [ ]:
# 3 epochs is a common default for fine-tuning. We could try a higher number of epochs as a further investigation.
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
)

predictions = trainer.predict(tokenized_test)

preds = np.argmax(predictions.predictions, axis=-1)

# The predict method calls compute_metrics which prints the classification report
print(preds)

Common methods for RoBERTa analysis

In [ ]:
from sklearn.metrics import f1_score
from sklearn.metrics import classification_report

# Common methods
def tokenize(batch):
    return tokenizer(batch['text'], padding='max_length', truncation=True, max_length=128)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    print(classification_report(labels, preds))
    return {'macro_f1': f1_score(labels, preds, average='macro')}

def prepare_dataset(split, task):
    tokenized = split.map(tokenize, batched=True)
    tokenized = tokenized.map(
        lambda examples: {'labels': [int(s) for s in examples[task]]},
        batched=True, remove_columns=[task]
    )
    tokenized.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])
    return tokenized

# 3 epochs is a common default for fine-tuning. We could try a higher number of epochs as a further investigation.
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1',
)

RoBERTA Sarcasm analysis

In [ ]:
# Sentiment
train_sentiment = prepare_dataset(dataset['train'], 'Sentiment')
test_sentiment = prepare_dataset(dataset['test'], 'Sentiment')
val_sentiment = prepare_dataset(dataset['validation'], 'Sentiment')

sentiment_model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2, problem_type="single_label_classification")

trainer = Trainer(
    model=sentiment_model,
    args=training_args,
    train_dataset=train_sentiment,
    eval_dataset=val_sentiment,
    compute_metrics=compute_metrics,
)
trainer.train()
predictions = trainer.predict(test_sentiment)



In [ ]:
# Sarcasm
train_sarcasm = prepare_dataset(dataset['train'], 'Sarcasm')
test_sarcasm = prepare_dataset(dataset['test'], 'Sarcasm')
val_sarcasm = prepare_dataset(dataset['validation'], 'Sarcasm')

sarcasm_model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2, problem_type="single_label_classification")

trainer_sarcasm = Trainer(
    model=sarcasm_model,
    args=training_args,
    train_dataset=train_sarcasm,
    eval_dataset=val_sarcasm,
    compute_metrics=compute_metrics,
)
trainer_sarcasm.train()
predictions_sarcasm = trainer_sarcasm.predict(test_sarcasm)

The results show a problem with the sarcasm model. Almost everything is being predicted as not sarcasm.

So the model needs adjusting to account for the class imbalance.

In [ ]:
from torch import nn
import torch

class WeightedTrainer(Trainer):
    # override the Trainer compute_loss
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        # model expects only the text inputs, so remove labels
        labels = inputs.pop("labels")
        # pass the text inputs through the model and get the logits outputs
        outputs = model(**inputs)
        logits = outputs.logits

        # divide the number of non-sarcasm examples by sarcasm examples for class 1 (sarcasm). This is about 6.15
        sarcasm_ratio = len(training_dataframe[training_dataframe['Sarcasm']==0]) / len(training_dataframe[training_dataframe['Sarcasm']==1])
        # create a weight tensor with two values, 1.0 for class 0 (not sarcasm) and the sarcasm_ratio for class 1
        weight = torch.tensor([1.0, sarcasm_ratio]).to(logits.device)
        loss_fn = nn.CrossEntropyLoss(weight=weight)
        # calculate the loss by comparing the model predictions against the actual labels
        loss = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss

trainer_sarcasm = WeightedTrainer(
    model=sarcasm_model,
    args=training_args,
    train_dataset=train_sarcasm,
    eval_dataset=val_sarcasm,
    compute_metrics=compute_metrics,
)

trainer_sarcasm.train()
predictions_sarcasm = trainer_sarcasm.predict(test_sarcasm)

The above approach didn't work well because it is now predicting too much sarcasm. So reduce the weighting to 3.

In [ ]:
from torch import nn
import torch

class WeightedTrainer(Trainer):
    # override the Trainer compute_loss
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        # model expects only the text inputs, so remove labels
        labels = inputs.pop("labels")
        # pass the text inputs through the model and get the logits outputs
        outputs = model(**inputs)
        logits = outputs.logits

        sarcasm_ratio = 3
        # create a weight tensor with two values, 1.0 for class 0 (not sarcasm) and the sarcasm_ratio for class 1
        weight = torch.tensor([1.0, sarcasm_ratio]).to(logits.device)
        loss_fn = nn.CrossEntropyLoss(weight=weight)
        # calculate the loss by comparing the model predictions against the actual labels
        loss = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss

# reduce the learning rate from the default 5e-5 to 1e-5
training_args_sarcasm = TrainingArguments(
    output_dir='./results_sarcasm',
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=1e-5,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1',
)

trainer_sarcasm = WeightedTrainer(
    model=sarcasm_model,
    args=training_args_sarcasm,
    train_dataset=train_sarcasm,
    eval_dataset=val_sarcasm,
    compute_metrics=compute_metrics,
)

trainer_sarcasm.train()
predictions_sarcasm = trainer_sarcasm.predict(test_sarcasm)

RoBERTa performs much better for sentiment, but only slightly better for sarcasm.

To confirm whether the lower LR is needed, run the 3x weight with the default LR:

In [ ]:
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1',
)

trainer_sarcasm = WeightedTrainer(
    model=sarcasm_model,
    args=training_args,
    train_dataset=train_sarcasm,
    eval_dataset=val_sarcasm,
    compute_metrics=compute_metrics,
)

trainer_sarcasm.train()
predictions_sarcasm = trainer_sarcasm.predict(test_sarcasm)

These results show degraded performance in the form of unstable training. The test set result of 0.62 is the same. But using a lower LR is preferred.

Let's try 2e-5

In [ ]:
# reduce the learning rate to 2e-5
training_args_sarcasm = TrainingArguments(
    output_dir='./results_sarcasm',
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1',
)

trainer_sarcasm = WeightedTrainer(
    model=sarcasm_model,
    args=training_args_sarcasm,
    train_dataset=train_sarcasm,
    eval_dataset=val_sarcasm,
    compute_metrics=compute_metrics,
)

trainer_sarcasm.train()
predictions_sarcasm = trainer_sarcasm.predict(test_sarcasm)

This shows improved results with the Sarcasm F1 matching LR (TF-IDF) F1 score. However this result is still worse than expected.

Now let's evaluate deBERTa

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
import numpy as np

model_name = 'microsoft/deberta-v3-base'
tokenizer = AutoTokenizer.from_pretrained(model_name)


train_sentiment = prepare_dataset(dataset['train'], 'Sentiment')
test_sentiment = prepare_dataset(dataset['test'], 'Sentiment')
val_sentiment = prepare_dataset(dataset['validation'], 'Sentiment')

sentiment_model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2, problem_type="single_label_classification")

trainer = Trainer(
    model=sentiment_model,
    args=training_args,
    train_dataset=train_sentiment,
    eval_dataset=val_sentiment,
    compute_metrics=compute_metrics,
)
trainer.train()
predictions = trainer.predict(test_sentiment)

In [ ]:
print('deBERTa Sarcasm')

# Sarcasm
train_sarcasm = prepare_dataset(dataset['train'], 'Sarcasm')
test_sarcasm = prepare_dataset(dataset['test'], 'Sarcasm')
val_sarcasm = prepare_dataset(dataset['validation'], 'Sarcasm')

sarcasm_model = AutoModelForSequenceClassification.from_pretrained('microsoft/deberta-v3-base', num_labels=2, problem_type="single_label_classification")

trainer_sarcasm = Trainer(
    model=sarcasm_model,
    args=training_args,
    train_dataset=train_sarcasm,
    eval_dataset=val_sarcasm,
    compute_metrics=compute_metrics,
)
trainer_sarcasm.train()
predictions_sarcasm = trainer_sarcasm.predict(test_sarcasm)

This suffers from the same class imbalance issue as RoBERTa. Will try the same solution.

In [ ]:

class DeBERTaWeightedTrainer(Trainer):
    # override the Trainer compute_loss
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        # model expects only the text inputs, so remove labels
        labels = inputs.pop("labels")
        # pass the text inputs through the model and get the logits outputs
        outputs = model(**inputs)
        logits = outputs.logits

        sarcasm_ratio = 3
        # create a weight tensor with two values, 1.0 for class 0 (not sarcasm) and the sarcasm_ratio for class 1
        # DeBERTa uses mixed precision float16/half datatype. so convert to this type
        print('converting tensor weights to ' + logits.dtype)
        weight = torch.tensor([1.0, 3.0]).to(logits.device).to(logits.dtype)
        loss_fn = nn.CrossEntropyLoss(weight=weight)
        loss = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss

sarcasm_model = AutoModelForSequenceClassification.from_pretrained('microsoft/deberta-v3-base', num_labels=2, problem_type="single_label_classification")

# reduce the learning rate to 2e-5
training_args_sarcasm = TrainingArguments(
    output_dir='./results_deBERTA_sarcasm',
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1',
)

trainer_sarcasm = DeBERTaWeightedTrainer(
    model=sarcasm_model,
    args=training_args_sarcasm,
    train_dataset=train_sarcasm,
    eval_dataset=val_sarcasm,
    compute_metrics=compute_metrics,
)

trainer_sarcasm.train()
predictions_sarcasm = trainer_sarcasm.predict(test_sarcasm)


This is still classifying everything as not sarcasm.